In [2]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_excel(file_name)

print("Rows:", df.shape[0], "Columns:", df.shape[1])
display(df.head())

arr = df[['Voltage', 'Current', 'Power']].to_numpy()
print("NumPy mean:", np.mean(arr, axis=0))
print("NumPy std:", np.std(arr, axis=0))

print("Activity 1: Information and Description")
df.info()
display(df.describe())

print("Activity 2: Null Values")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print("Total nulls:", nulls.sum())

print("Activity 3: Handling Missing Values")
df.fillna({'Wh': df['Wh'].median(), 'Chamber_Temp_degC': df['Ambient_Setpoint_C']}, inplace=True)
df.fillna({'Temp_Differential_C': df['Battery_Temp_degC'] - df['Chamber_Temp_degC']}, inplace=True)
df.fillna(0, inplace=True)
print("Total nulls after handling:", df.isnull().sum().sum())

print("Activity 4: Sorting and Top Records")
display(df.sort_values('Voltage', ascending=False).head(5)[['TimeStamp', 'Voltage', 'Current', 'SoC_%']])
display(df[df['Current'] > 0].sort_values('Power', ascending=False).head(8)[['TimeStamp', 'Current', 'Power']])

print("Activity 5: Frequency Listing")
print(df['Cycle_Name'].value_counts())
print(df['Ambient_Setpoint_C'].value_counts())

print("Activity 6: Sorting Rows and Columns, Indexing")
display(df.sort_index(ascending=False).head(3))
display(df.sort_index(axis=1).head(3))
display(df.iloc[0:5, 0:4])
df_time = df.set_index('TimeStamp').sort_index()
display(df_time.loc['2017-03-19':'2017-03-20', ['Voltage', 'Current']].head())

print("Activity 7: Conditional Access")
display(df.loc[(df['Current'] > 2) & (df['SoC_%'] < 30), ['Voltage', 'Current', 'SoC_%']].head())
display(df.loc[(df['Ambient_Setpoint_C'] == -20) & (df['Battery_Temp_degC'] < -15) & (df['Voltage'] < 3.2), ['Voltage', 'Battery_Temp_degC']].head())
display(df.loc[((df['Ambient_Setpoint_C'] == 25) & (df['High_Load_Flag'] == 1)) | (df['Power'] < -25), ['Ambient_Setpoint_C', 'High_Load_Flag', 'Power']].head())

print("Activity 8: Minimum and Maximum")
display(df[['Voltage', 'Current', 'Battery_Temp_degC', 'SoC_%']].agg(['min', 'max']))
display(df.loc[[df['Battery_Temp_degC'].idxmax()], ['TimeStamp', 'Battery_Temp_degC', 'Ambient_Setpoint_C']])

print("Activity 9: Group By")
print(df.groupby('Ambient_Setpoint_C')['Voltage'].mean())
print(df.groupby(['Ambient_Setpoint_C', 'High_Load_Flag'])['Battery_Temp_degC'].mean())

print("Activity 10: Adding New Columns")
df['Temp_Rise_C'] = df['Battery_Temp_degC'] - df['Ambient_Setpoint_C']
df['Mode'] = np.where(df['Current'] > 0, 'Charging', np.where(df['Current'] < 0, 'Discharging', 'Idle'))
display(df[['Battery_Temp_degC', 'Ambient_Setpoint_C', 'Temp_Rise_C', 'Current', 'Mode']].head())

print("Activity 11: Group By with Aggregates")
display(df.groupby('Cycle_Name').agg(avg_voltage=('Voltage', 'mean'), max_temp=('Battery_Temp_degC', 'max')).round(3))
display(df.groupby(['Ambient_Setpoint_C', 'Mode']).agg({'Voltage': ['mean', 'min', 'max'], 'Power': 'sum'}).round(3))

print("Activity 12: Selection on Groups")
display(df.groupby('Ambient_Setpoint_C').get_group(-20)[['Voltage', 'Current', 'Battery_Temp_degC']].head())
display(df[df['Cycle_Name'] == 'UDDS_Pan18650PF'][['Voltage', 'Current', 'SoC_%']].head())

print("Activity 13: Correlation")
print("Voltage vs SoC_%:", df['Voltage'].corr(df['SoC_%']))
print("Current vs Power:", df['Current'].corr(df['Power']))
display(df[['Voltage', 'Current', 'Power', 'Battery_Temp_degC', 'SoC_%']].corr().round(3))

print("Activity 14: Normalization")
cols = ['Voltage', 'Current', 'Battery_Temp_degC', 'SoC_%']
minmax = (df[cols] - df[cols].min()) / (df[cols].max() - df[cols].min())
zscore = (df[cols] - df[cols].mean()) / df[cols].std()
display(minmax.agg(['min', 'max']))
display(zscore.agg(['mean', 'std']).round(3))

print("Activity 15: Joining, Merging and Concatenation")
info = pd.DataFrame({'Ambient_Setpoint_C': [-20, -10, 0, 10, 25], 'Condition': ['Extreme Cold', 'Very Cold', 'Freezing', 'Cool', 'Room Temp']})
display(pd.merge(df, info, on='Ambient_Setpoint_C', how='left')[['Ambient_Setpoint_C', 'Condition', 'Voltage']].head())
display(df.join(info.set_index('Ambient_Setpoint_C'), on='Ambient_Setpoint_C')[['Ambient_Setpoint_C', 'Condition']].head())
display(pd.concat([df.head(3), df.tail(3)], axis=0)[['TimeStamp', 'Voltage', 'Current']])
display(pd.concat([df[['Voltage']].head(), df[['Current']].head()], axis=1))

print("Post Lab 1: Broadcasting")
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(a + np.array([10, 20, 30]))
print(a * np.array([[1], [2], [3]]))
print(a * 2)

print("Post Lab 2: loc vs iloc")
demo = df[['Voltage', 'Current', 'SoC_%']].head(4).copy()
demo.index = ['A', 'B', 'C', 'D']
print(demo.loc['B'])
print(demo.iloc[1])
print(demo.loc['A':'C', 'Voltage'])
print(demo.iloc[0:3, 0])

Saving panasonic_engineered_sample.xlsx to panasonic_engineered_sample (1).xlsx
Rows: 50000 Columns: 24


,TimeStamp,Voltage,Current,Ah,Wh,Power,Battery_Temp_degC,Time,Chamber_Temp_degC,Ambient_Setpoint_C,...,dV_dt,dI_dt,dT_dt,Voltage_roll_mean,Voltage_roll_std,Current_roll_mean,Current_roll_std,Battery_Temp_degC_roll_mean,Battery_Temp_degC_roll_std,High_Load_Flag
0,2017-04-09 22:00:15,3.51342,-1.20615,-1.61142,-5.53350,-4.237712,12.910318,273902.743999,NaN,10,...,-0.091906,-0.902204,0.000000,3.531340,0.045286,-0.479162,1.134518,12.911940,0.004060,0
1,2017-06-11 23:41:57,2.96592,-2.22121,-2.01555,NaN,-6.587931,-3.355205,399743.016997,0.0,-10,...,0.000000,0.072776,0.119875,3.089789,0.090927,-1.462485,0.754407,-3.361743,0.006095,0
2,2017-03-19 05:59:13,3.40791,-1.64876,-2.23061,-7.97942,-5.618826,25.854860,9243.368994,25.0,25,...,0.000000,-0.007664,0.000000,3.405539,0.002863,-1.762150,0.103921,25.886248,0.081736,0
3,2017-04-06 10:03:39,3.40083,-1.36049,-1.97728,-7.22564,-4.626795,12.504952,199142.037999,NaN,10,...,0.000000,0.000000,0.000000,3.387411,0.010283,-1.657510,0.251117,12.501709,0.005254,0
4,2017-06-26 10:00:32,3.59399,0.00000,-0.93231,-3.08375,0.000000,-15.621599,210272.638991,NaN,-20,...,0.006737,0.000000,0.000000,3.576743,0.020516,-0.027522,0.063939,-15.626630,0.005733,0


NumPy mean: [ 3.60107946 -0.97909291 -3.30894171]
NumPy std: [0.28116995 1.81765482 6.16510791]
Activity 1: Information and Description
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   TimeStamp                    50000 non-null  datetime64[ns]
 1   Voltage                      50000 non-null  float64       
 2   Current                      50000 non-null  float64       
 3   Ah                           50000 non-null  float64       
 4   Wh                           42182 non-null  float64       
 5   Power                        50000 non-null  float64       
 6   Battery_Temp_degC            50000 non-null  float64       
 7   Time                         50000 non-null  float64       
 8   Chamber_Temp_degC            19270 non-null  float64       
 9   Ambient_Setpoint_C           50000 

,TimeStamp,Voltage,Current,Ah,Wh,Power,Battery_Temp_degC,Time,Chamber_Temp_degC,Ambient_Setpoint_C,...,dV_dt,dI_dt,dT_dt,Voltage_roll_mean,Voltage_roll_std,Current_roll_mean,Current_roll_std,Battery_Temp_degC_roll_mean,Battery_Temp_degC_roll_std,High_Load_Flag
count,50000,50000.000000,50000.000000,50000.000000,42182.000000,50000.000000,50000.000000,50000.000000,19270.000000,50000.000000,...,49998.000000,49998.000000,49998.000000,50000.000000,49999.000000,50000.000000,49999.000000,50000.000000,49999.000000,50000.000000
mean,2017-05-04 05:48:19.629420032,3.601079,-0.979093,-1.163598,-4.248172,-3.308942,7.410392,69305.216930,14.853140,4.290000,...,-0.000370,0.013568,-0.000468,3.601105,0.030724,-0.984879,0.538286,7.412195,0.050659,0.023940
min,2017-03-18 02:17:45,2.492400,-20.635170,-2.798170,-9.709240,-53.347965,-20.331744,0.000000,0.000000,-20.000000,...,-2.713988,-102.462006,-2.603179,2.497109,0.000000,-13.251111,0.000000,-20.331744,0.000000,0.000000
25%,2017-03-28 18:26:55.500000,3.431710,-1.697955,-1.724197,-6.240110,-6.049352,-5.400930,3893.450754,0.000000,-10.000000,...,-0.006214,-0.008453,0.000000,3.432216,0.005442,-1.687676,0.104321,-5.398509,0.005362,0.000000
50%,2017-04-09 20:53:46.500000,3.606220,-0.508750,-1.121215,-4.121520,-1.852236,10.987729,8290.793001,25.000000,10.000000,...,0.000000,0.000000,0.000000,3.605897,0.018615,-0.645145,0.358995,10.992363,0.043415,0.000000
75%,2017-06-11 16:33:37.750000128,3.804220,-0.026130,-0.557452,-2.126653,-0.099548,15.064529,22061.400500,25.000000,10.000000,...,0.006154,0.007941,0.000000,3.802046,0.042039,-0.071406,0.743804,14.974425,0.087833,0.000000
max,2017-06-26 21:26:20,4.322960,9.381830,2.504720,9.884650,35.367519,32.759120,586314.163002,26.000000,25.000000,...,5.749179,160.359657,4.328357,4.199992,0.526861,7.130862,8.522763,32.778846,12.219959,1.000000
std,NaN,0.281173,1.817673,0.710727,2.530232,6.165170,13.997243,121732.903967,12.273312,15.062883,...,0.106838,3.294862,0.503642,0.277324,0.037485,1.623312,0.636629,13.995555,0.175267,0.152864


Activity 2: Null Values
Wh                             7818
Chamber_Temp_degC             30730
Temp_Differential_C           30730
dV_dt                             2
dI_dt                             2
dT_dt                             2
Voltage_roll_std                  1
Current_roll_std                  1
Battery_Temp_degC_roll_std        1
dtype: int64
Total nulls: 69287
Activity 3: Handling Missing Values
Total nulls after handling: 0
Activity 4: Sorting and Top Records


...

,Ambient_Setpoint_C,Condition,Voltage
0,10,Cool,3.51342
1,-10,Very Cold,2.96592
2,25,Room Temp,3.40791
3,10,Cool,3.40083
4,-20,Extreme Cold,3.59399


,Ambient_Setpoint_C,Condition
0,10,Cool
1,-10,Very Cold
2,25,Room Temp
3,10,Cool
4,-20,Extreme Cold


,TimeStamp,Voltage,Current
0,2017-04-09 22:00:15,3.51342,-1.20615
1,2017-06-11 23:41:57,2.96592,-2.22121
2,2017-03-19 05:59:13,3.40791,-1.64876
49997,2017-06-12 05:34:19,3.96652,0.00000
49998,2017-06-12 14:20:01,3.61957,-1.95254
49999,2017-03-21 00:52:20,4.10871,0.57003


,Voltage,Current
0,3.51342,-1.20615
1,2.96592,-2.22121
2,3.40791,-1.64876
3,3.40083,-1.36049
4,3.59399,0.00000


Post Lab 1: Broadcasting
[[11 22 33]
 [14 25 36]
 [17 28 39]]
[[ 1  2  3]
 [ 8 10 12]
 [21 24 27]]
[[ 2  4  6]
 [ 8 10 12]
 [14 16 18]]
Post Lab 2: loc vs iloc
Voltage     2.965920
Current    -2.221210
SoC_%      30.498276
Name: B, dtype: float64
Voltage     2.965920
Current    -2.221210
SoC_%      30.498276
Name: B, dtype: float64
A    3.51342
B    2.96592
C    3.40791
Name: Voltage, dtype: float64
A    3.51342
B    2.96592
C    3.40791
Name: Voltage, dtype: float64
